# CSE 251B — v2 Bundle (SOAP + RoPE/RMSNorm/SwiGLU/QK-norm + 70/30 mix + WSD, 15k steps)

Goal: push val.bin PPL from ~42 to sub-30 by closing the OOD gap and finishing training.

Bundled changes vs `colab_optimizer_ablation.ipynb` SOAP run:

| Change | Why |
|---|---|
| **70/30 FineWeb-Edu + vanilla FineWeb** | Hidden test is multi-domain. Pure edu under-generalizes. |
| **15,000 steps** (was 5,000) | 96M params at 1.97B tokens ≈ Chinchilla optimal. |
| **RoPE** (drop `wpe`) | Free param recovery, better extrapolation. |
| **`bias=False`** on all Linears | Modern default; tiny capacity recovered. |
| **RMSNorm** | Standard upgrade over LayerNorm. |
| **SwiGLU MLP** | Llama-style gated MLP at matched param count. |
| **QK-norm** | Stabilizes attention; lets us push LR. |
| **WSD schedule** | Outperforms cosine at matched compute (Keller Jordan records). |
| **SOAP optimizer** | Your ablation winner. Locked in. |

Architecture geometry (`n_layer=8, n_embd=768`) is unchanged so the delta is attributable to the bundle, not shape. Submission cell (last) writes `model.py` + `checkpoint.pt` + `config.json` that drop directly into the HF repo and pass the evaluator's `model(idx) -> logits` contract.

**Runtime**: ~9-12 hr on T4, ~3-4 hr on A100. Checkpointing every 500 steps — safe to disconnect/resume.


In [ ]:
# ── 1. GPU check + install ─────────────────────────────────────────────────────
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or '⚠️ No GPU')

import torch
print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  {p.total_memory/1e9:.1f} GB')

!pip install -q huggingface_hub wandb heavyball

In [ ]:
# ── 2. Config ────────────────────────────────────────────────────────────────────────────────
from pathlib import Path
import math

RUN_NAME = 'v2_bundle'

USE_DRIVE = True  # @param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/cse251b')
else:
    BASE_DIR = Path('/content/cse251b')

EDU_DIR  = BASE_DIR / 'data' / 'finewebedu10B'
WEB_DIR  = BASE_DIR / 'data' / 'fineweb10B'
CKPT_DIR = BASE_DIR / f'checkpoints_{RUN_NAME}'
SUBMIT_DIR = BASE_DIR / f'submission_{RUN_NAME}'
for d in (EDU_DIR, WEB_DIR, CKPT_DIR, SUBMIT_DIR): d.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / 'latest.pt'

# ── Training schedule ───────────────────────────────────────────────────────────────────────
MICRO_BATCH = 8
SEQ_LEN     = 1024
GRAD_ACCUM  = 16            # effective batch = 131,072 tokens/step
NUM_STEPS   = 15000         # 3× longer than v1 → ~1.97B tokens, ~Chinchilla-optimal for 96M
GRAD_CLIP   = 1.0
VAL_EVERY   = 500
CKPT_EVERY  = 500

# ── Data mixing ───────────────────────────────────────────────────────────────────────────────────
EDU_RATIO   = 0.70          # 70% FineWeb-Edu, 30% vanilla FineWeb
NUM_EDU_SHARDS = 10         # ~1B edu tokens
NUM_WEB_SHARDS = 6          # ~600M general-web tokens (we sample 30% of mix)

# ── SOAP hyperparams ─────────────────────────────────────────────────────────────────────────────
SOAP_LR     = 3e-3
SOAP_MIN_LR = 0.0           # WSD decays to zero, not 10% of peak
SOAP_WD     = 0.1
SOAP_BETAS  = (0.95, 0.95)
SOAP_PRECOND_FREQ = 10

# ── WSD schedule ────────────────────────────────────────────────────────────────────────────────
WARMUP_STEPS  = 500         # ~3% warmup
COOLDOWN_FRAC = 0.30        # last 30% linear decay to SOAP_MIN_LR

# ── W&B ───────────────────────────────────────────────────────────────────────────────────────────────
USE_WANDB     = False  # @param {type:"boolean"}
WANDB_PROJECT = 'cse251b-group-gli'
WANDB_RUN     = RUN_NAME

print(f'Run        : {RUN_NAME}')
print(f'Total steps: {NUM_STEPS:,}  (~{NUM_STEPS * MICRO_BATCH * SEQ_LEN * GRAD_ACCUM / 1e9:.2f}B tokens)')
print(f'Mix        : {int(EDU_RATIO*100)}% edu / {int((1-EDU_RATIO)*100)}% web')
print(f'Ckpt dir   : {CKPT_DIR}')

In [ ]:
# ── 3. Download both datasets (skip if cached) ───────────────────────────────────────────────────
from huggingface_hub import hf_hub_download

EDU_REPO = 'kjj0/finewebedu10B-gpt2'
WEB_REPO = 'kjj0/fineweb10B-gpt2'   # vanilla FineWeb (general web, multi-domain)

def maybe_download(repo, fname, dest_dir):
    dest = dest_dir / fname
    if dest.exists(): print(f'  [cached] {fname}'); return
    print(f'  ↓ {fname}', end='', flush=True)
    hf_hub_download(repo_id=repo, filename=fname, repo_type='dataset', local_dir=str(dest_dir))
    print(' ✓')

print('FineWeb-Edu:')
maybe_download(EDU_REPO, 'finewebedu_val_000000.bin', EDU_DIR)
for i in range(1, NUM_EDU_SHARDS + 1):
    maybe_download(EDU_REPO, f'finewebedu_train_{i:06d}.bin', EDU_DIR)

print('\nFineWeb (vanilla):')
# NOTE: if this repo name doesn't exist or has different file naming, swap to whatever
# pre-tokenized multi-domain shards you have access to. Karpathy's build-nanogpt also
# produces compatible shards. The shard format must be: 1024-byte header (magic=20240520,
# version=1, n_tokens) followed by uint16 token IDs.
try:
    maybe_download(WEB_REPO, 'fineweb_val_000000.bin', WEB_DIR)
    for i in range(1, NUM_WEB_SHARDS + 1):
        maybe_download(WEB_REPO, f'fineweb_train_{i:06d}.bin', WEB_DIR)
except Exception as e:
    print(f'⚠️  vanilla FineWeb download failed: {e}')
    print('   Edit WEB_REPO/filenames above to point at any compatible shard set.')
    raise

edu_train = sorted(EDU_DIR.glob('finewebedu_train_*.bin'))
edu_val   = sorted(EDU_DIR.glob('finewebedu_val_*.bin'))
web_train = sorted(WEB_DIR.glob('fineweb_train_*.bin'))
print(f'\nEdu  train {len(edu_train)} shards   val {len(edu_val)} shards')
print(f'Web  train {len(web_train)} shards')
assert edu_train and edu_val and web_train

In [ ]:
# ── 4. Mixed data loader (Bernoulli per draw) ──────────────────────────────────────────────────────────
import numpy as np

def load_shard(path):
    with open(path, 'rb') as f:
        hdr = np.frombuffer(f.read(256 * 4), dtype=np.int32)
        assert hdr[0] == 20240520 and hdr[1] == 1
        return np.frombuffer(f.read(int(hdr[2]) * 2), dtype=np.uint16).copy()

class MixedDataLoader:
    """Two independent streams; each draw flips Bernoulli(EDU_RATIO) to pick source.
    GRAD_ACCUM=16 means each optimizer step sees ~70%/30% draws in expectation."""
    def __init__(self, edu_files, web_files, B, T, edu_ratio, seed=42):
        self.B, self.T = B, T
        self.edu_ratio = edu_ratio
        self.streams = {'edu': {'files': list(edu_files), 'fi': 0, 'pos': 0, 'tok': None},
                        'web': {'files': list(web_files), 'fi': 0, 'pos': 0, 'tok': None}}
        for name, s in self.streams.items():
            s['tok'] = load_shard(s['files'][0])
        self.rng = np.random.default_rng(seed)

    def _draw(self, name):
        s = self.streams[name]; B, T = self.B, self.T
        while s['pos'] + B * T + 1 > len(s['tok']):
            s['fi'] = (s['fi'] + 1) % len(s['files'])
            s['tok'] = load_shard(s['files'][s['fi']]); s['pos'] = 0
        buf = s['tok'][s['pos'] : s['pos'] + B * T + 1]; s['pos'] += B * T
        x = torch.from_numpy(buf[:-1].astype(np.int64)).view(B, T).cuda()
        y = torch.from_numpy(buf[1:].astype(np.int64)).view(B, T).cuda()
        return x, y

    def __iter__(self): return self
    def __next__(self):
        return self._draw('edu' if self.rng.random() < self.edu_ratio else 'web')

    def state_dict(self):
        return {n: {'fi': s['fi'], 'pos': s['pos']} for n, s in self.streams.items()}
    def load_state_dict(self, st):
        for n, ss in st.items():
            self.streams[n]['fi']  = ss['fi']
            self.streams[n]['tok'] = load_shard(self.streams[n]['files'][ss['fi']])
            self.streams[n]['pos'] = ss['pos']

class SingleDataLoader:
    """For validation — a single stream, no mixing."""
    def __init__(self, files, B, T):
        self.files, self.B, self.T = list(files), B, T
        self._fi, self._pos = 0, 0
        self._tok = load_shard(self.files[0])
    def __iter__(self): return self
    def __next__(self):
        B, T = self.B, self.T
        while self._pos + B*T + 1 > len(self._tok):
            self._fi = (self._fi + 1) % len(self.files)
            self._tok = load_shard(self.files[self._fi]); self._pos = 0
        buf = self._tok[self._pos : self._pos + B*T + 1]; self._pos += B*T
        x = torch.from_numpy(buf[:-1].astype(np.int64)).view(B, T).cuda()
        y = torch.from_numpy(buf[1:].astype(np.int64)).view(B, T).cuda()
        return x, y

In [ ]:
# ── 5. Model: RoPE + RMSNorm + SwiGLU + QK-norm, bias=False ────────────────────────────────────────────
# This entire cell is re-emitted to disk in the submission cell so model.py stays in sync.

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        return F.rms_norm(x, (x.size(-1),), self.weight, self.eps)

class Rotary(nn.Module):
    """Half-truncated RoPE (modded-nanogpt style): first half rotated, second half passthrough."""
    def __init__(self, head_dim, base_freq_div=1024.0):
        super().__init__()
        angular_freq = (1.0 / base_freq_div) ** torch.linspace(0, 1, steps=head_dim // 4, dtype=torch.float32)
        self.register_buffer('angular_freq',
                             torch.cat([angular_freq, angular_freq.new_zeros(head_dim // 4)]))
    def forward(self, x_BTHD):
        pos = torch.arange(x_BTHD.size(1), dtype=torch.float32, device=x_BTHD.device)
        theta = torch.outer(pos, self.angular_freq)[None, :, None, :]
        cos, sin = theta.cos(), theta.sin()
        x1, x2 = x_BTHD.to(torch.float32).chunk(2, dim=-1)
        y1 = x1 * cos + x2 * sin
        y2 = -x1 * sin + x2 * cos
        return torch.cat((y1, y2), dim=3).type_as(x_BTHD)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head   = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.rotary = Rotary(self.head_dim)
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim)
        k = k.view(B, T, self.n_head, self.head_dim)
        v = v.view(B, T, self.n_head, self.head_dim)
        # QK-norm
        q = F.rms_norm(q, (self.head_dim,))
        k = F.rms_norm(k, (self.head_dim,))
        # RoPE
        q = self.rotary(q); k = self.rotary(k)
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    """SwiGLU at matched param count vs nanoGPT's 4x GELU (hidden = 8/3 * dim)."""
    def __init__(self, dim):
        super().__init__()
        hidden = int(dim * 8 / 3)
        hidden = (hidden + 63) // 64 * 64  # round up to multiple of 64 for tensor cores
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.n_embd)
        self.attn  = CausalSelfAttention(config)
        self.norm2 = RMSNorm(config.n_embd)
        self.mlp   = SwiGLU(config.n_embd)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp (self.norm2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304   # padded for tensor cores
    n_layer:    int = 8
    n_head:     int = 12
    n_embd:     int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte    = nn.Embedding(config.vocab_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.norm_f = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight   # tied embeddings
        self.apply(self._init_weights)
        n = sum(p.numel() for p in self.parameters())
        assert n <= 100_000_000, f'Over 100M: {n:,}'
        print(f'Parameters: {n/1e6:.2f}M')

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for block in self.blocks: x = block(x)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        if targets is not None:
            # cross-entropy over the full padded 50304 — targets are in [0, 50257), padded entries unused
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        # Inference path: trim to GPT-2's exact 50257 (evaluator contract)
        return logits[..., :50257]

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

In [ ]:
# ── 6. SOAP + WSD schedule ──────────────────────────────────────────────────────────────────────────────
def make_wsd_schedule(peak_lr, min_lr, total_steps, warmup_steps, cooldown_frac):
    """Warmup→Stable→Decay. Linear warmup, hold at peak, linear decay to min_lr in last cooldown_frac.
    Outperforms cosine at matched compute in modded-nanogpt records."""
    cooldown_start = int(total_steps * (1 - cooldown_frac))
    def get_lr(step):
        if step < warmup_steps:
            return peak_lr * (step + 1) / warmup_steps
        if step < cooldown_start:
            return peak_lr
        if step >= total_steps:
            return min_lr
        frac = (step - cooldown_start) / (total_steps - cooldown_start)
        return peak_lr + (min_lr - peak_lr) * frac
    return get_lr

def safe_exp(x):
    try: return math.exp(x)
    except OverflowError: return float('inf')

print('SOAP + WSD ready.')

In [ ]:
# ── 7. Build model + SOAP + resume ──────────────────────────────────────────────────────────────────────
import time, inspect

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = GPTConfig()
model  = GPT(config).to(device)

try:
    compiled = torch.compile(model, dynamic=False)
    print('torch.compile: enabled')
except Exception as e:
    compiled = model
    print(f'torch.compile: disabled ({e})')

decay_params   = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
nodecay_params = [p for p in model.parameters() if p.requires_grad and p.dim() <  2]

from heavyball import SOAP
optimizer = SOAP(
    [{'params': decay_params,   'weight_decay': SOAP_WD},
     {'params': nodecay_params, 'weight_decay': 0.0}],
    lr=SOAP_LR, betas=SOAP_BETAS, precondition_frequency=SOAP_PRECOND_FREQ,
)
lr_schedule = make_wsd_schedule(SOAP_LR, SOAP_MIN_LR, NUM_STEPS, WARMUP_STEPS, COOLDOWN_FRAC)
print(f'SOAP: {sum(p.numel() for p in decay_params):,} decay + '
      f'{sum(p.numel() for p in nodecay_params):,} nodecay params')

for grp in optimizer.param_groups:
    grp['initial_lr'] = grp['lr']

train_loader = MixedDataLoader(edu_train, web_train, MICRO_BATCH, SEQ_LEN, EDU_RATIO)
val_loader   = SingleDataLoader(edu_val, 16, SEQ_LEN)

start_step, train_losses, val_log = 0, [], []
if CKPT_PATH.exists():
    print(f'\nResuming from {CKPT_PATH}')
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    train_loader.load_state_dict(ckpt['loader'])
    start_step   = ckpt['step']
    train_losses = ckpt.get('train_losses', [])
    val_log      = ckpt.get('val_log', [])
    print(f'Resumed at step {start_step}/{NUM_STEPS}')

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, name=WANDB_RUN, resume='allow',
               config=dict(run=RUN_NAME, n_layer=config.n_layer, n_embd=config.n_embd,
                           micro_batch=MICRO_BATCH, grad_accum=GRAD_ACCUM,
                           num_steps=NUM_STEPS, edu_ratio=EDU_RATIO,
                           total_params=model.get_num_params()))
    print('W&B initialized.')

In [ ]:
# ── 8. Training loop ──────────────────────────────────────────────────────────────────────────────────────────
compiled.train()
t_start = time.time(); t_step = time.time()
tokens_seen = start_step * MICRO_BATCH * SEQ_LEN * GRAD_ACCUM

for step in range(start_step, NUM_STEPS + 1):
    # ── Validation ─────────────────────────────────────────────────────────────────────────────
    if step % VAL_EVERY == 0 or step == NUM_STEPS:
        compiled.eval()
        vl = 0.0
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            for _ in range(20):
                _, loss = compiled(*next(val_loader))
                vl += loss.item()
        val_loss = vl / 20; val_ppl = safe_exp(val_loss)
        elapsed = time.time() - t_start
        tok_s = (step - start_step) * MICRO_BATCH * SEQ_LEN * GRAD_ACCUM / max(elapsed, 1)
        print(f'step {step:5d}/{NUM_STEPS}  val_loss {val_loss:.4f}  '
              f'val_ppl {val_ppl:.2f}  tok/s {tok_s:.0f}  {elapsed/60:.1f}min')
        val_log.append((step, val_loss, val_ppl))
        if USE_WANDB:
            wandb.log({'val_loss': val_loss, 'val_ppl': val_ppl, 'tokens': tokens_seen}, step=step)
        compiled.train()

    if step == NUM_STEPS: break

    # ── Checkpoint ─────────────────────────────────────────────────────────────────────────────
    if step > 0 and step % CKPT_EVERY == 0:
        torch.save({'step': step, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'loader': train_loader.state_dict(),
                    'train_losses': train_losses, 'val_log': val_log}, CKPT_PATH)
        print(f'  ✓ checkpoint at step {step}')

    # ── LR ───────────────────────────────────────────────────────────────────────────────────────────
    lr_now = lr_schedule(step)
    for grp in optimizer.param_groups: grp['lr'] = lr_now

    # ── Train step ───────────────────────────────────────────────────────────────────────────────
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.0
    for _ in range(GRAD_ACCUM):
        X, Y = next(train_loader)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            _, loss = compiled(X, Y)
            (loss / GRAD_ACCUM).backward()
        loss_sum += loss.item() / GRAD_ACCUM
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()

    tokens_seen += MICRO_BATCH * SEQ_LEN * GRAD_ACCUM
    train_losses.append(loss_sum)

    if step % 50 == 0 and step > 0:
        step_ms = (time.time() - t_step) / 50 * 1000
        print(f'  step {step:5d}  train_loss {loss_sum:.4f}  lr {lr_now:.2e}  {step_ms:.0f}ms/step')
        t_step = time.time()
        if USE_WANDB:
            wandb.log({'train_loss': loss_sum, 'lr': lr_now, 'tokens': tokens_seen}, step=step)

print('\nTraining complete!')

In [ ]:
# ── 9. Full validation PPL (FineWeb-Edu val — your internal metric) ─────────────────────────────────────────────────
compiled.eval()
val_tokens = load_shard(edu_val[0])
EVAL_BATCH = 32
total_loss, total_toks = 0.0, 0
with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
    for i in range(0, len(val_tokens) - EVAL_BATCH * SEQ_LEN, EVAL_BATCH * SEQ_LEN):
        chunk = val_tokens[i : i + EVAL_BATCH * SEQ_LEN + 1]
        if len(chunk) < EVAL_BATCH * SEQ_LEN + 1: break
        X = torch.from_numpy(chunk[:-1].astype(np.int64)).view(EVAL_BATCH, SEQ_LEN).cuda()
        Y = torch.from_numpy(chunk[1:].astype(np.int64)).view(EVAL_BATCH, SEQ_LEN).cuda()
        _, loss = compiled(X, Y)
        total_loss += loss.item() * EVAL_BATCH * SEQ_LEN
        total_toks += EVAL_BATCH * SEQ_LEN
avg_loss  = total_loss / total_toks
final_ppl = safe_exp(avg_loss)
print(f'Run                   : {RUN_NAME}')
print(f'Val loss (full shard) : {avg_loss:.4f}')
print(f'Val PPL  (full shard) : {final_ppl:.2f}')
print(f'Total params          : {model.get_num_params():,} ({model.get_num_params()/1e6:.1f}M)')
print(f'\nNOTE: this is FineWeb-Edu val. Contest val.bin is multi-domain; expect a +5-10 PPL gap.')

In [ ]:
# ── 10. Write submission files (model.py + checkpoint.pt + config.json) ──────────────
# Evaluator contract:
#   - model(input_ids) -> logits (NOT a tuple, exactly 50257 wide)
#   - load_model(checkpoint_path, device) -> nn.Module in eval mode
#
# IMPORTANT: MODEL_SRC below is a hand-copy of cell 5. If you edit cell 5, mirror
# the change here. (inspect.getsource doesn't work on classes defined in a Colab cell.)

import json

MODEL_SRC = r'''class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        return F.rms_norm(x, (x.size(-1),), self.weight, self.eps)

class Rotary(nn.Module):
    """Half-truncated RoPE (modded-nanogpt style): first half rotated, second half passthrough."""
    def __init__(self, head_dim, base_freq_div=1024.0):
        super().__init__()
        angular_freq = (1.0 / base_freq_div) ** torch.linspace(0, 1, steps=head_dim // 4, dtype=torch.float32)
        self.register_buffer("angular_freq",
                             torch.cat([angular_freq, angular_freq.new_zeros(head_dim // 4)]))
    def forward(self, x_BTHD):
        pos = torch.arange(x_BTHD.size(1), dtype=torch.float32, device=x_BTHD.device)
        theta = torch.outer(pos, self.angular_freq)[None, :, None, :]
        cos, sin = theta.cos(), theta.sin()
        x1, x2 = x_BTHD.to(torch.float32).chunk(2, dim=-1)
        y1 = x1 * cos + x2 * sin
        y2 = -x1 * sin + x2 * cos
        return torch.cat((y1, y2), dim=3).type_as(x_BTHD)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head   = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.rotary = Rotary(self.head_dim)
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim)
        k = k.view(B, T, self.n_head, self.head_dim)
        v = v.view(B, T, self.n_head, self.head_dim)
        q = F.rms_norm(q, (self.head_dim,))
        k = F.rms_norm(k, (self.head_dim,))
        q = self.rotary(q); k = self.rotary(k)
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    """SwiGLU at matched param count vs nanoGPT's 4x GELU (hidden = 8/3 * dim)."""
    def __init__(self, dim):
        super().__init__()
        hidden = int(dim * 8 / 3)
        hidden = (hidden + 63) // 64 * 64
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.n_embd)
        self.attn  = CausalSelfAttention(config)
        self.norm2 = RMSNorm(config.n_embd)
        self.mlp   = SwiGLU(config.n_embd)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp (self.norm2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer:    int = 8
    n_head:     int = 12
    n_embd:     int = 768

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte    = nn.Embedding(config.vocab_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.norm_f = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        n = sum(p.numel() for p in self.parameters())
        assert n <= 100_000_000, f"Over 100M: {n:,}"
        print(f"Parameters: {n/1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for block in self.blocks: x = block(x)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        return logits[..., :50257]

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())
'''

torch.save(model.state_dict(), SUBMIT_DIR / "checkpoint.pt")

config_dict = {
    "block_size": config.block_size, "vocab_size": config.vocab_size,
    "n_layer": config.n_layer, "n_head": config.n_head, "n_embd": config.n_embd,
    "val_ppl_internal": final_ppl,
    "total_params": model.get_num_params(),
    "run_name": RUN_NAME,
}
with open(SUBMIT_DIR / "config.json", "w") as f:
    json.dump(config_dict, f, indent=2)

model_py = f'''# model.py — CSE 251B group GLI (v2 bundle submission)
# Required interface: load_model(checkpoint_path, device) -> nn.Module
# Forward contract: model(input_ids) -> logits of shape (B, T, 50257)

import json, math
from dataclasses import dataclass
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

{MODEL_SRC}

def load_model(checkpoint_path, device):
    cfg_path = Path(checkpoint_path).parent / "config.json"
    with open(cfg_path) as f: c = json.load(f)
    cfg = GPTConfig(block_size=c["block_size"], vocab_size=c["vocab_size"],
                    n_layer=c["n_layer"], n_head=c["n_head"], n_embd=c["n_embd"])
    m = GPT(cfg).to(device)
    state = torch.load(checkpoint_path, map_location=device)
    if "model" in state: state = state["model"]
    m.load_state_dict(state)
    m.eval()
    n = sum(p.numel() for p in m.parameters())
    assert n <= 100_000_000
    print(f"Loaded v2 bundle: {{n:,}} params ({{n/1e6:.1f}}M)")
    return m
'''

(SUBMIT_DIR / "model.py").write_text(model_py)

print(f"Wrote submission to {SUBMIT_DIR}:")
for f in sorted(SUBMIT_DIR.iterdir()):
    print(f"  {f.name:20s}  {f.stat().st_size/1e6:>8.2f} MB")
print()
print("Next:")
print(f"  1. Verify locally:  python evaluate.py --model_dir {SUBMIT_DIR} --data val.bin")
print(f"  2. Push to HF:      huggingface-cli upload <repo> {SUBMIT_DIR}/checkpoint.pt {SUBMIT_DIR}/model.py {SUBMIT_DIR}/config.json")
